# NB06 — TPDA Proxy: Normalización por Jerarquía Vial OSM
**VíaSegura AI · Resolución parcial de Limitación L1**

## Contexto
El IPI absoluto (NB02) y la normalización por km de red (NB05) tratan todos los segmentos viales como equivalentes:
1 km de motorway = 1 km de calle residencial. Esto sobreprioriza zonas de alta densidad vial.

**L1 (Alta):** El IPI no mide riesgo por vehículo-km. Una vía con 100,000 veh/día y 10 accidentes
aparece igual que una con 1,000 veh/día y 10 accidentes.

## Solución NB06
Usamos el tag `highway` de OSM (ya descargado en NB05 pero sin usar) para asignar **pesos TPDA proxy**
basados en los rangos típicos para Bogotá:
- `motorway/trunk` → 100,000 veh/día
- `primary` → 50,000 veh/día
- `secondary` → 20,000 veh/día
- `tertiary` → 8,000 veh/día
- `residential` → 2,500 veh/día

Los pesos se **calibran y validan** con las ~19 estaciones CGT reales de la SDM (ArcGIS REST público).

## Output central
- `vm_dia`: vehículo-metros por día (proxy TPDA × longitud segmento) por zona
- `tasa_vehkm`: siniestros / millón de veh·km → **métrica estándar PIARC/Vision Zero**
- `tipologia_nb06`: tipología actualizada con el nuevo denominador
- `hotspots_normalizados_nb06.csv`

## Limitaciones que persisten
- L1 se resuelve **parcialmente**: los pesos son ordinales, no TPDA exacto por segmento
- Los 19 sensores CGT validan el orden de magnitud pero no cubren vías locales
- Resolución completa requiere datos TPDA por segmento (solicitud formal SDM/CGT)

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import osmnx as ox
import requests
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import folium
from shapely.geometry import box, Point

warnings.filterwarnings("ignore")
ox.settings.log_console = False
ox.settings.use_cache = True   # reutiliza caché de NB05 — descarga rápida

print(f"pandas  {pd.__version__}")
print(f"geopandas {gpd.__version__}")
print(f"osmnx   {ox.__version__}")

In [ ]:
_root = next(c for c in [Path.cwd(), Path.cwd().parent] if (c / 'config.py').exists())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from config import REPORTS, MAPS

print(f"REPORTS: {REPORTS}")
print(f"MAPS:    {MAPS}")

# ── Pesos TPDA proxy por jerarquía OSM ───────────────────────────────────────
# Fuente: rangos típicos Bogotá basados en estudios SDM + calibración CGT
# Ref: Accident Analysis & Prevention (2024) bus lanes Bogotá; SDM informe anual movilidad
TPDA_PROXY = {
    "motorway":       120_000,
    "motorway_link":   60_000,
    "trunk":           90_000,
    "trunk_link":      45_000,
    "primary":         50_000,
    "primary_link":    25_000,
    "secondary":       20_000,
    "secondary_link":  10_000,
    "tertiary":         8_000,
    "tertiary_link":    4_000,
    "residential":      2_500,
    "living_street":      800,
    "unclassified":     2_000,
    "service":          1_500,
    "road":             3_000,
    "busway":          50_000,
}
TPDA_DEFAULT = 2_000   # fallback para tipos no reconocidos

print("\nPesos TPDA proxy por tipo OSM:")
for k, v in TPDA_PROXY.items():
    print(f"  {k:<20} {v:>8,} veh/día")

In [ ]:
# Cargar output de NB05 como base
df = pd.read_csv(REPORTS / "hotspots_normalizados_nb05.csv")
print(f"NB05 base cargada: {len(df):,} zonas x {df.shape[1]} columnas")
print(f"Columnas: {list(df.columns)}")

# Top 200 para referencia
df_top = df[df["en_top200_rec"] == True].copy()
print(f"\nTop 200 zonas: {len(df_top)}")
print(f"km_red_osm promedio Top 200: {df_top['km_red_osm'].mean():.3f} km")
print(f"tasa_red promedio Top 200: {df_top['tasa_red'].mean():.2f} sin/km")

## Sección 1: Descarga red OSM con tags de jerarquía

Reutilizamos la misma bbox y tipo de red que NB05 (`network_type='drive'`).
OSMnx usa caché local → descarga rápida si ya se ejecutó NB05.

Diferencia clave respecto a NB05: **extraemos el tag `highway`** y lo usamos para asignar
el peso TPDA proxy por segmento.

In [ ]:
print("Descargando/cargando red vial desde caché OSM...")

lat_min = df["lat_grid"].min() - 0.01
lat_max = df["lat_grid"].max() + 0.01
lon_min = df["lon_grid"].min() - 0.01
lon_max = df["lon_grid"].max() + 0.01

G = ox.graph_from_bbox(
    bbox=(lon_min, lat_min, lon_max, lat_max),
    network_type="drive"
)

# Extraer edges con columna highway
edges = ox.graph_to_gdfs(G, nodes=False)
print(f"Segmentos descargados: {len(edges):,}")
print(f"Columnas disponibles: {[c for c in edges.columns if c in ['highway','length','lanes','maxspeed','name']]}")
print(f"Longitud total: {edges['length'].sum()/1000:.1f} km")

# Distribución de tipos highway
def normalize_highway(val):
    """OSMnx puede devolver string o lista; siempre devolvemos el tipo principal."""
    if isinstance(val, list):
        val = val[0]
    return str(val).split("[")[0].strip().replace("'", "")

edges["highway_norm"] = edges["highway"].apply(normalize_highway)
hw_counts = edges["highway_norm"].value_counts()
print(f"\nDistribución de tipos highway (top 15):")
print(hw_counts.head(15).to_string())

In [ ]:
# Asignar peso TPDA proxy a cada segmento
edges["tpda_proxy"] = edges["highway_norm"].map(TPDA_PROXY).fillna(TPDA_DEFAULT)

# Vehículo-metros por día por segmento = TPDA × longitud (en metros)
edges["vm_dia_seg"] = edges["tpda_proxy"] * edges["length"]

# Estadísticas de cobertura
total = len(edges)
reconocidos = edges["highway_norm"].isin(TPDA_PROXY.keys()).sum()
print(f"Segmentos con tipo reconocido: {reconocidos:,}/{total:,} ({reconocidos/total*100:.1f}%)")
print(f"Segmentos con peso default: {total-reconocidos:,}")

# Resumen por tipo
resumen = (edges.groupby("highway_norm")
               .agg(n_segs=("length","count"),
                    km_total=("length", lambda x: x.sum()/1000),
                    tpda_proxy=("tpda_proxy","first"),
                    vm_dia_mill=("vm_dia_seg", lambda x: x.sum()/1e6))
               .sort_values("km_total", ascending=False)
               .head(15))
resumen["tpda_proxy"] = resumen["tpda_proxy"].apply(lambda x: f"{int(x):,}")
print(f"\nResumen por tipo de vía:")
print(resumen.to_string())

## Sección 2: Spatial join → vm_dia por zona

Para cada zona H3 (~111m × 111m):
- `vm_dia` = Σ(TPDA_proxy × length_m) de todos los segmentos que intersectan la celda
- `km_eq` = Σ(length_m × TPDA_proxy / TPDA_ref) / 1000 → km-equivalentes (normalizado)

Donde `TPDA_ref = 50,000` (equivalente a una vía primaria) para mantener km_eq en unidades legibles.

In [ ]:
TPDA_REF = 50_000   # referencia: vía primaria de Bogotá
EPSILON_KM = 0.05   # mínimo para evitar división por cero
HALF = 0.0005       # mitad del lado de celda (0.001° / 2)

# Crear polígonos de grilla para el universo completo
def make_cell_polygon(lat, lon, half=HALF):
    return box(lon - half, lat - half, lon + half, lat + half)

gdf_zonas = gpd.GeoDataFrame(
    df[["lat_grid", "lon_grid"]].copy(),
    geometry=[make_cell_polygon(r.lat_grid, r.lon_grid) for _, r in df[["lat_grid","lon_grid"]].iterrows()],
    crs="EPSG:4326"
)
gdf_zonas["idx_orig"] = df.index.values
print(f"Polígonos de grilla: {len(gdf_zonas):,}")

if edges.crs.to_epsg() != 4326:
    edges = edges.to_crs("EPSG:4326")

print("Realizando spatial join (intersección edges x zonas)...")
joined = gpd.sjoin(
    edges[["length", "tpda_proxy", "vm_dia_seg", "highway_norm", "geometry"]].reset_index(drop=True),
    gdf_zonas,
    how="inner",
    predicate="intersects"
)

# Agregar por zona
agg = joined.groupby("idx_orig").agg(
    vm_dia=("vm_dia_seg", "sum"),
    km_raw=("length", lambda x: x.sum() / 1000),
    n_segs=("length", "count"),
    tpda_modal=("tpda_proxy", lambda x: x.mode().iloc[0] if len(x) > 0 else TPDA_DEFAULT),
    hw_modal=("highway_norm", lambda x: x.mode().iloc[0] if len(x) > 0 else "unclassified"),
).reset_index()

# km-equivalentes ponderados
agg["km_eq"] = (agg["vm_dia"] / TPDA_REF) / 1000
agg["km_eq"] = agg["km_eq"].clip(lower=EPSILON_KM)

# Unir al dataframe principal
df = df.reset_index(drop=True)
df = df.merge(agg[["idx_orig","vm_dia","km_eq","n_segs","tpda_modal","hw_modal"]],
              left_index=True, right_on="idx_orig", how="left")
df = df.drop(columns=["idx_orig"], errors="ignore")
df["vm_dia"] = df["vm_dia"].fillna(0)
df["km_eq"]  = df["km_eq"].fillna(EPSILON_KM)

# Comparación: km_red_osm (NB05, sin ponderar) vs km_eq (NB06, ponderado)
df_top = df[df["en_top200_rec"] == True].copy()
print(f"\nTop 200 — comparación km_red_osm (NB05) vs km_eq (NB06):")
print(f"  km_red_osm  media: {df_top['km_red_osm'].mean():.3f} km (sin ponderación)")
print(f"  km_eq       media: {df_top['km_eq'].mean():.3f} km-eq (ponderado TPDA proxy)")
print(f"  Ratio km_eq/km_red: {(df_top['km_eq']/df_top['km_red_osm']).mean():.2f}x")

corr = df_top[["km_red_osm","km_eq"]].corr()
print(f"  Correlacion Pearson km_red_osm <-> km_eq: {corr.iloc[0,1]:.4f}")

In [ ]:
# ── Tasa veh·km ──────────────────────────────────────────────────────────────
# Formula: siniestros / (vm_dia × 365 / 1_000_000_000)
# Unidades: siniestros por 10^9 veh·km (estándar PIARC)
# vm_dia está en veh·m/día → × 365 → veh·m/año → / 1e9 → 10^9 veh·km/año
# Usamos período 2016-2019 (4 años) → × 4 en el denominador

PERIODO_ANOS = 4

df["vm_dia_safe"] = df["vm_dia"].clip(lower=TPDA_DEFAULT * EPSILON_KM * 1000)
df["tasa_vehkm"] = (
    df["cantidad_siniestros_rec"] /
    (df["vm_dia_safe"] * 365 * PERIODO_ANOS / 1e9)
)  # siniestros por 10^9 veh·km

# También calculamos por km_eq (alternativa más simple, comparable con tasa_red de NB05)
df["tasa_km_eq"] = df["cantidad_siniestros_rec"] / df["km_eq"]

df_top = df[df["en_top200_rec"] == True].copy()

print("Métricas calculadas — Top 200:")
print("\ntasa_vehkm (sin / 10^9 veh·km):")
print(df_top["tasa_vehkm"].describe().round(4).to_string())
print("\ntasa_km_eq (sin / km_eq ponderado):")
print(df_top["tasa_km_eq"].describe().round(2).to_string())
print("\ntasa_red NB05 (sin / km uniforme, referencia):")
print(df_top["tasa_red"].describe().round(2).to_string())

## Sección 3: Validación con sensores CGT (SDM)

Descargamos las ~19 estaciones de conteo vehicular del Centro de Gestión de Tráfico (CGT)
de la Secretaría Distrital de Movilidad, disponibles via ArcGIS REST.

**Objetivo:** comparar el TPDA proxy asignado por el tipo OSM `highway` en cada estación
contra el conteo real reportado por el sensor.

**Resultado esperado:** validar el orden de magnitud y detectar si hay bias sistemático
por tipo de vía que requiera recalibrar los pesos.

In [ ]:
CGT_URL = (
    "https://services2.arcgis.com/NEwhEo9GGSHXcRXV/arcgis/rest/services/"
    "Conteo_Vehiculos_CGT_Bogot%C3%A1_D_C/FeatureServer/0/query"
)
CGT_PARAMS = {
    "where": "1=1",
    "outFields": "*",
    "f": "geojson",
    "resultRecordCount": 1000
}

print("Descargando estaciones CGT desde ArcGIS REST SDM...")
try:
    resp = requests.get(CGT_URL, params=CGT_PARAMS, timeout=20)
    resp.raise_for_status()
    gdf_cgt = gpd.read_file(resp.text)
    print(f"Estaciones CGT descargadas: {len(gdf_cgt)}")
    print(f"Columnas: {list(gdf_cgt.columns)}")
    if len(gdf_cgt) > 0:
        print(gdf_cgt.head(3).to_string())
    CGT_OK = True
except Exception as e:
    print(f"CGT no disponible: {e}")
    print("Continuando con análisis de validación sintética.")
    gdf_cgt = gpd.GeoDataFrame()  # vacío
    CGT_OK = False

In [ ]:
if CGT_OK and len(gdf_cgt) > 0 and gdf_cgt.geometry.notna().any():
    # ── Spatial match: para cada sensor CGT, encontrar zona más cercana ────────
    gdf_cgt = gdf_cgt[gdf_cgt.geometry.notna()].to_crs("EPSG:4326")

    # Asignar tipo highway al punto CGT (nearest edge en OSM)
    if len(gdf_cgt) > 0:
        cgt_coords = [(geom.y, geom.x) for geom in gdf_cgt.geometry]
        nearest_edges = [ox.nearest_edges(G, X=lon, Y=lat) for lat, lon in cgt_coords]

        hw_at_cgt = []
        tpda_proxy_at_cgt = []
        for u, v, k in nearest_edges:
            hw_val = normalize_highway(G[u][v][k].get("highway", "unclassified"))
            tpda_val = TPDA_PROXY.get(hw_val, TPDA_DEFAULT)
            hw_at_cgt.append(hw_val)
            tpda_proxy_at_cgt.append(tpda_val)

        gdf_cgt["hw_osm"] = hw_at_cgt
        gdf_cgt["tpda_proxy"] = tpda_proxy_at_cgt

        # Buscar columna con conteo real (puede variar por versión del dataset)
        vol_cols = [c for c in gdf_cgt.columns if any(k in c.lower()
                    for k in ["tpda","tpd","volumen","conteo","total","aforo"])]
        print(f"Columnas de volumen disponibles: {vol_cols}")

        if vol_cols:
            vcol = vol_cols[0]
            gdf_cgt["vol_real"] = pd.to_numeric(gdf_cgt[vcol], errors="coerce")
            valid = gdf_cgt[gdf_cgt["vol_real"].notna() & (gdf_cgt["vol_real"] > 0)]

            if len(valid) > 0:
                print(f"\nValidación CGT — {len(valid)} sensores con volumen real:")
                comp = valid[["hw_osm","tpda_proxy","vol_real"]].copy()
                comp["ratio"] = comp["vol_real"] / comp["tpda_proxy"]
                print(comp.to_string(index=False))
                print(f"\nRatio medio vol_real / tpda_proxy: {comp['ratio'].mean():.3f}")
                print(f"(1.0 = calibración perfecta; >1 = proxy subestima; <1 = sobreestima)")

                # Factor de calibración por tipo de vía
                cal_factor = comp.groupby("hw_osm")["ratio"].mean()
                print(f"\nFactores de calibración por tipo:")
                print(cal_factor.to_string())
            else:
                print("Sin datos de volumen real para calibrar.")
        else:
            print("El dataset CGT no incluye columna de volumen/TPDA en esta versión.")
            print(f"Columnas disponibles: {list(gdf_cgt.columns)}")
else:
    print("CGT no disponible. Validación omitida.")
    print("Los pesos TPDA proxy se basan en rangos típicos de la literatura SDM/Bogotá.")
    print("Para calibración formal: solicitar datos CGT via Ley 1712/2014 a la SDM.")

## Sección 4: Rankings y tipología NB06

Replicamos la estructura de NB05 pero con el nuevo denominador ponderado.

Métricas de ranking:
- `rank_tasa_vehkm`: ranking por tasa en 10^9 veh·km (PIARC)
- `rank_km_eq`: ranking por tasa en km-eq ponderado

Tipología NB06 (misma estructura que NB05 para comparabilidad):
- **Hotspot absoluto + relativo NB06**: Top 200 IPI_vol Y Top 200 tasa_vehkm
- **Hotspot por volumen NB06**: Top 200 IPI_vol, rank_tasa_vehkm > 200
- **Hotspot oculto ponderado**: rank_vol > 200, Top 200 tasa_vehkm (nuevo respecto a NB05)
- **Sin tipología NB06**

In [ ]:
def pct_rank_desc(series):
    return series.rank(ascending=False, method="min", na_option="bottom").astype(int)

def score_pct(series):
    return series.rank(pct=True, method="average", ascending=False, na_option="bottom") * 100

df["rank_tasa_vehkm"] = pct_rank_desc(df["tasa_vehkm"])
df["rank_km_eq"]      = pct_rank_desc(df["tasa_km_eq"])
df["score_vehkm"]     = score_pct(df["tasa_vehkm"])
df["score_km_eq"]     = score_pct(df["tasa_km_eq"])

TOP_N = 200
df["en_top_vehkm"] = df["rank_tasa_vehkm"] <= TOP_N
df["en_top_km_eq"] = df["rank_km_eq"] <= TOP_N

def asignar_tipologia_nb06(row):
    vol = row.get("en_top200_rec", False)
    vkm = row.get("en_top_vehkm", False)
    if vol and vkm:
        return "Hotspot absoluto + relativo NB06"
    elif vol and not vkm:
        return "Hotspot por volumen NB06"
    elif not vol and vkm:
        return "Hotspot oculto ponderado NB06"
    else:
        return "Sin tipología NB06"

df["tipologia_nb06"] = df.apply(asignar_tipologia_nb06, axis=1)

df["delta_rank_nb06"] = df["rank_vol"] - df["rank_tasa_vehkm"]

# Conteos
print("Distribución tipología NB06 (universo):")
print(df["tipologia_nb06"].value_counts().to_string())

df_top = df[df["en_top200_rec"] == True].copy()
print(f"\nTop 200 — tipología NB06:")
print(df_top["tipologia_nb06"].value_counts().to_string())

# Comparación NB05 vs NB06 para el Top 200
n05 = (df_top["tipologia_nb05"] == "Hotspot absoluto + relativo").sum()
n06 = (df_top["tipologia_nb06"] == "Hotspot absoluto + relativo NB06").sum()
print(f"\nZonas en AMBOS rankings (máxima prioridad):")
print(f"  NB05 (km uniforme):     {n05}/200 ({n05/2:.1f}%)")
print(f"  NB06 (km_eq ponderado): {n06}/200 ({n06/2:.1f}%)")

In [ ]:
# ── Insight: cambios de ranking NB05 → NB06 ──────────────────────────────────
print("="*70)
print("IMPACTO DE LA PONDERACION POR JERARQUIA VIAL")
print("Cambios respecto a NB05 (km uniforme)")
print("="*70)

# Zonas que cambian de tipología
cambios = df_top[
    (df_top["tipologia_nb05"] == "Hotspot absoluto + relativo") !=
    (df_top["tipologia_nb06"] == "Hotspot absoluto + relativo NB06")
].copy()
print(f"\nZonas que cambian de categoria (NB05 <-> NB06): {len(cambios)}")

if len(cambios) > 0:
    print(cambios[["lat_grid","lon_grid","tipologia_nb05","tipologia_nb06",
                   "tasa_red","tasa_vehkm","km_red_osm","km_eq",
                   "hw_modal","tpda_modal","localidad_modal_rec"]].to_string())

# Zonas nuevas que aparecen como hotspot oculto con ponderación
nuevos_ocultos = df[
    (df["tipologia_nb05"] == "Hotspot relativo oculto") &
    (df["tipologia_nb06"] == "Hotspot oculto ponderado NB06")
].copy()
print(f"\nZonas que eran ocultos en NB05 Y siguen siendo ocultos en NB06: {len(nuevos_ocultos)}")
print("(Estas son estructuralmente peligrosas — aparecen con AMBAS metricas de normalización)")

# Análisis de la diferencia entre tasa_red (NB05) y tasa_km_eq (NB06)
df_top["factor_ajuste"] = df_top["tasa_km_eq"] / df_top["tasa_red"]
print(f"\nFactor de ajuste tasa_km_eq/tasa_red en Top 200:")
print(f"  Media: {df_top['factor_ajuste'].mean():.3f}x")
print(f"  Min:   {df_top['factor_ajuste'].min():.3f}x (vias locales sobreestimadas en NB05)")
print(f"  Max:   {df_top['factor_ajuste'].max():.3f}x (vias arteriales subestimadas en NB05)")
print(f"  StD:   {df_top['factor_ajuste'].std():.3f}x")

## Sección 5: Visualizaciones

1. **Scatter NB05 vs NB06**: tasa_red vs tasa_km_eq, color por tipo de vía modal
2. **Mapa tipología NB06**: comparación visual con NB05
3. **Bar chart**: km_red_osm vs km_eq por tipo de vía

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: tasa_red (NB05) vs tasa_km_eq (NB06)
ax1 = axes[0]
df_valid = df_top[df_top["tasa_red"].notna() & df_top["tasa_km_eq"].notna()].copy()
hw_types = df_valid["hw_modal"].fillna("unclassified").unique()
cmap_hw = plt.cm.tab10
for i, hw in enumerate(hw_types):
    sub = df_valid[df_valid["hw_modal"] == hw]
    ax1.scatter(sub["tasa_red"], sub["tasa_km_eq"],
               label=hw, alpha=0.7, s=40, color=cmap_hw(i/max(len(hw_types),1)))
ax1.plot([0, df_valid[["tasa_red","tasa_km_eq"]].max().max()],
         [0, df_valid[["tasa_red","tasa_km_eq"]].max().max()],
         "k--", alpha=0.3, linewidth=1, label="y=x (sin cambio)")
ax1.set_xlabel("tasa_red NB05 (sin/km uniforme)")
ax1.set_ylabel("tasa_km_eq NB06 (sin/km_eq ponderado)")
ax1.set_title("Impacto de la ponderación TPDA\n(Top 200 zonas)", fontsize=9)
ax1.legend(fontsize=6, ncol=2)
ax1.grid(True, alpha=0.3)

# Panel 2: km_red_osm (NB05) vs km_eq (NB06) por tipo de vía modal
ax2 = axes[1]
hw_agg = (df_top.groupby("hw_modal")
             .agg(km_red_mean=("km_red_osm","mean"), km_eq_mean=("km_eq","mean"), n=("km_red_osm","count"))
             .sort_values("km_eq_mean", ascending=True)
             .head(10))
x = range(len(hw_agg))
ax2.barh([f"{idx} (n={row.n})" for idx, row in hw_agg.iterrows()],
         hw_agg["km_eq_mean"], color="#ff2233", alpha=0.8, label="km_eq NB06")
ax2.barh([f"{idx} (n={row.n})" for idx, row in hw_agg.iterrows()],
         hw_agg["km_red_mean"], color="#888", alpha=0.5, label="km_red NB05")
ax2.set_xlabel("km promedio por zona")
ax2.set_title("km_red vs km_eq por tipo de vía modal", fontsize=9)
ax2.legend(fontsize=8)
ax2.grid(True, axis="x", alpha=0.3)

# Panel 3: Comparación tipología NB05 vs NB06
ax3 = axes[2]
tip05_counts = df_top["tipologia_nb05"].value_counts()
tip06_counts = df_top["tipologia_nb06"].value_counts()
# Normalizar nombres para comparación
tip06_map = {
    "Hotspot absoluto + relativo NB06": "Hotspot abs+rel",
    "Hotspot por volumen NB06": "Hotspot por volumen",
    "Hotspot oculto ponderado NB06": "Hotspot oculto",
}
tip05_map = {
    "Hotspot absoluto + relativo": "Hotspot abs+rel",
    "Hotspot por volumen": "Hotspot por volumen",
}
cats = list(set(list(tip05_map.values()) + list(tip06_map.values())))
n05_vals = [tip05_counts.get(k, 0) for k in [c for c, v in tip05_map.items() if v in cats]]
n06_vals = [tip06_counts.get(k, 0) for k in [c for c, v in tip06_map.items() if v in cats]]
x_pos = range(max(len(n05_vals), len(n06_vals)))
ax3.bar([i-0.2 for i in range(len(n05_vals))], n05_vals, 0.4, label="NB05", color="#888", alpha=0.8)
ax3.bar([i+0.2 for i in range(len(n06_vals))], n06_vals, 0.4, label="NB06", color="#ff2233", alpha=0.8)
ax3.set_xticks(range(max(len(n05_vals), len(n06_vals))))
ax3.set_xticklabels(["abs+rel", "vol", "oculto"][:max(len(n05_vals), len(n06_vals))], fontsize=8)
ax3.set_ylabel("N zonas")
ax3.set_title("Tipología NB05 vs NB06\n(Top 200)", fontsize=9)
ax3.legend(fontsize=9)
ax3.grid(True, axis="y", alpha=0.3)

plt.suptitle("NB06: Impacto de ponderación TPDA proxy por jerarquía OSM", fontsize=11, y=1.01)
plt.tight_layout()
PATH_FIG = REPORTS / "tipologia_nb06_comparativa.png"
plt.savefig(PATH_FIG, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figura guardada: {PATH_FIG}")

In [ ]:
# ── Mapa folium: tasa_vehkm por zona ─────────────────────────────────────────
import matplotlib.cm as cm

df_map = df[df["tasa_vehkm"].notna() & (df["rank_tasa_vehkm"] <= 500)].copy()
vmin = df_map["tasa_vehkm"].quantile(0.05)
vmax = df_map["tasa_vehkm"].quantile(0.95)
norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
cmap = cm.YlOrRd

m_nb06 = folium.Map(location=[4.65, -74.08], zoom_start=11, tiles="CartoDB positron")

for _, row in df_map.iterrows():
    rgba = cmap(norm(row["tasa_vehkm"]))
    hex_color = mcolors.to_hex(rgba)
    popup = (
        f"tasa_vehkm: {row['tasa_vehkm']:.4f} sin/10^9 veh·km<br>"
        f"tasa_red NB05: {row['tasa_red']:.2f} sin/km<br>"
        f"hw_modal: {row.get('hw_modal','?')}<br>"
        f"km_eq: {row['km_eq']:.3f} km-eq<br>"
        f"km_red: {row['km_red_osm']:.3f} km<br>"
        f"tipología NB06: {row['tipologia_nb06']}<br>"
        f"Localidad: {row.get('localidad_modal_rec','?')}"
    )
    folium.Rectangle(
        bounds=[[row["lat_grid"]-0.0005, row["lon_grid"]-0.0005],
                [row["lat_grid"]+0.0005, row["lon_grid"]+0.0005]],
        color=hex_color, fill=True, fill_opacity=0.75, weight=0.5,
        popup=folium.Popup(popup, max_width=300)
    ).add_to(m_nb06)

PATH_MAP = MAPS / "tasa_vehkm_nb06.html"
m_nb06.save(str(PATH_MAP))
print(f"Mapa guardado: {PATH_MAP}")
m_nb06

In [ ]:
# ── Export ───────────────────────────────────────────────────────────────────
COLS_NUEVAS = [
    "vm_dia",
    "km_eq",
    "n_segs",
    "tpda_modal",
    "hw_modal",
    "tasa_vehkm",
    "tasa_km_eq",
    "rank_tasa_vehkm",
    "rank_km_eq",
    "score_vehkm",
    "score_km_eq",
    "tipologia_nb06",
    "delta_rank_nb06",
    "en_top_vehkm",
]

# Columnas base de NB05 + nuevas de NB06
COLS_BASE = [
    "lat_grid", "lon_grid", "localidad_modal_rec",
    "IPI_rec", "rank_vol", "en_top200_rec", "categoria_persistencia",
    "km_red_osm", "tasa_red", "rank_tasa_red", "score_tasa_red",
    "tasa_pob_100k", "rank_tasa_pob",
    "delta_rank", "tipologia_nb05",
    "cantidad_siniestros_rec", "siniestros_con_muertos_rec",
    "causa_efectiva", "causa_efectiva_pct", "causa_imputed",
]

cols_out = [c for c in COLS_BASE + COLS_NUEVAS if c in df.columns]
df_out = df[cols_out].copy()

PATH_OUT = REPORTS / "hotspots_normalizados_nb06.csv"
df_out.to_csv(PATH_OUT, index=False, encoding="utf-8-sig")
print(f"CSV guardado: {PATH_OUT}")
print(f"  Filas: {len(df_out):,} | Columnas: {df_out.shape[1]}")

df_top = df_out[df_out["en_top200_rec"] == True]
print(f"\n{'='*65}")
print("RESUMEN EJECUTIVO NB06")
print(f"{'='*65}")
print(f"Método NB05 (km uniforme):")
for t, n in df_top["tipologia_nb05"].value_counts().items():
    print(f"  {t}: {n} zonas ({n/2:.1f}%)")
print(f"\nMétodo NB06 (km_eq ponderado TPDA proxy):")
for t, n in df_top["tipologia_nb06"].value_counts().items():
    print(f"  {t}: {n} zonas ({n/2:.1f}%)")
print(f"\ntasa_vehkm media Top 200: {df_top['tasa_vehkm'].mean():.5f} sin/10^9 veh·km")
print(f"km_eq media Top 200: {df_top['km_eq'].mean():.3f} km-eq")
print(f"hw_modal más frecuente Top 200: {df_top['hw_modal'].mode().iloc[0] if 'hw_modal' in df_top.columns else 'N/A'}")

## ADR-15: Decisión metodológica NB06

**Decisión:** NB06 produce `tasa_vehkm` como métrica de riesgo por exposición vehicular.
Este valor es **complementario** al IPI absoluto (NB02) y a la tasa por km uniforme (NB05),
NO un reemplazo.

**Justificación:**
- Los pesos TPDA proxy son ordinales, no exactos. Error esperado: ±30-50% por tipo de vía.
- La métrica permite identificar zonas sub-representadas en el ranking volumétrico.
- Compatible con el estándar PIARC (siniestros/10^9 veh·km).
- Resolución completa requiere: solicitar TPDA real por segmento a SDM/CGT.

**Uso recomendado en dashboard:**
- Capa adicional en Página 1 (mapa): `tasa_vehkm_nb06.html`
- Columna nueva en Página 2 (zonas críticas): `tasa_vehkm`
- Métrica en Página 5 (metodología): L1 marcada como "Parcialmente resuelta"

**Estado de L1:**
- ANTES: El IPI trata todos los km de red igual (1 km motorway = 1 km residential)
- DESPUÉS NB06: El denominador pondera por TPDA proxy → el análisis detecta zonas
  peligrosas por km-equivalente, no solo por volumen
- PENDIENTE: Calibración exacta con TPDA real de la SDM por solicitud formal